# Imports

In [1]:
import os
import json
import joblib
from pathlib import Path

In [2]:
import numpy as np
import pandas as pd

In [3]:
data_store=Path("./Prepared_Data_Store")

# Loading Models

In [94]:
model1=joblib.load(os.path.join(data_store,"model1.pkl"))

model4=joblib.load(os.path.join(data_store,"model4.pkl"))

model2=joblib.load(os.path.join(data_store,"model2.pkl"))

model3=joblib.load(os.path.join(data_store,"model3.pkl"))

model8=joblib.load(os.path.join(data_store,"model8.pkl"))

# Loading Model's Predictors (X Data)

## Model-1

In [4]:
model1_train_X=pd.read_csv(
    os.path.join(data_store,"model1_train_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

model1_test_X=pd.read_csv(
    os.path.join(data_store,"model1_test_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

In [5]:
model1_X=pd.concat([model1_train_X,model1_test_X],axis=0).sort_index()

## Model-4

In [6]:
model4_train_X=pd.read_csv(
    os.path.join(data_store,"model4_train_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

model4_test_X=pd.read_csv(
    os.path.join(data_store,"model4_test_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

In [7]:
model4_X=pd.concat([model4_train_X,model4_test_X],axis=0).sort_index()

## Model-2

In [8]:
model2_train_X=pd.read_csv(
    os.path.join(data_store,"model2_train_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

model2_test_X=pd.read_csv(
    os.path.join(data_store,"model2_test_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

In [9]:
model2_X=pd.concat([model2_train_X,model2_test_X],axis=0).sort_index()

## Model-3

In [10]:
model3_train_X=pd.read_csv(
    os.path.join(data_store,"model3_train_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

model3_test_X=pd.read_csv(
    os.path.join(data_store,"model3_test_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

In [11]:
model3_X=pd.concat([model3_train_X,model3_test_X],axis=0).sort_index()

## Model-8

### Note:

#### 1. During the Data Preparation of the Model8 we removed the first entry of training and testing set.
#### 2. So we have to include these entries from the original features data.
#### 3. And also we have make the non stationary features stationary by first order difference.
#### 4. So for the missing entries we will take the first order difference from the respective median of train and test respectively.

### Loading Train_X And Test_X

In [13]:
model8_train_X=pd.read_csv(
    os.path.join(data_store,"model8_train_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

model8_test_X=pd.read_csv(
    os.path.join(data_store,"model8_test_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

In [14]:
model8_X=pd.concat([model8_train_X,model8_test_X],axis=0).sort_index()

In [72]:
tickers=list(model8_X.index.get_level_values("ticker").unique())

In [ ]:
idx=pd.IndexSlice

### Loading Non Stationary Features

In [12]:
with open(os.path.join(data_store,"non_stationary_features.json"),"r") as file:
    non_stationary_features=json.load(file)

### Missing Train Entry

In [15]:
missing_train_entry_date=model1_train_X.index.get_level_values("date")[0]

In [31]:
non_stat_features_train_median=(model1_train_X
                              .groupby("ticker")[non_stationary_features]
                              .apply(lambda x: x.median(axis=0)))

In [65]:
missing_train_entry=model1_train_X.loc[idx[missing_train_entry_date,:]]

In [68]:
missing_train_entry[non_stationary_features]=(
    missing_train_entry[non_stationary_features]-
    non_stat_features_train_median
    
)

In [70]:
missing_train_entry["date"]=missing_train_entry_date
missing_train_entry.reset_index(inplace=True)
missing_train_entry.set_index(["date","ticker"],inplace=True)

In [71]:
missing_train_entry

open      high       low     close    volume  \
date       ticker                                                     
2010-02-22 A      -1.461939 -1.511306 -1.462434 -1.482303 -0.156541   
           ACGL   -1.689435 -1.692413 -1.671041 -1.689585  3.054903   
           ACN    -1.579930 -1.597301 -1.588901 -1.596231 -0.338131   
           ADI    -1.392832 -1.401546 -1.400281 -1.395146  0.904611   
           ADM    -0.602551 -0.608262 -0.582109 -0.597555 -0.511968   
...                     ...       ...       ...       ...       ...   
           XEL    -1.493121 -1.499792 -1.499390 -1.503222 -0.549615   
           XOM    -1.982734 -2.023290 -2.002735 -2.035675  0.447519   
           YUM    -2.494197 -2.515981 -2.495412 -2.512273 -0.265248   
           ZBH    -0.720712 -0.731881 -0.720414 -0.728635 -0.907416   
           ZBRA   -0.642211 -0.654275 -0.648073 -0.656495 -0.668779   

                   dollar_volume  dollar_volume_7d  dollar_volume_15d  \
date       ticker                                                       
2010-02-22 A           -0.693067         -0.279116          -0.341175   
           ACGL         1.343749          1.911222           1.708494   
           ACN         -0.758531         -0.957332          -0.950981   
           ADI         -0.044209          1.348163           0.612157   
           ADM         -0.916221         -0.615638           0.118269   
...                          ...               ...                ...   
           XEL         -0.938931         -1.100931          -0.956601   
           XOM         -0.202423          0.454527           1.278541   
           YUM         -0.862111         -1.190647          -1.041378   
           ZBH         -1.157855         -1.157903          -0.841791   
           ZBRA        -0.665587         -0.579336          -0.565859   

                   dollar_volume_21d  dollar_volume_rank  ...  max_price_7d  \
date       ticker                                         ...                 
2010-02-22 A               -0.708899            0.014026  ...     -1.578164   
           ACGL             1.505118           -2.431023  ...     -1.706761   
           ACN             -0.963062            1.277944  ...     -1.613072   
           ADI              0.524131           -1.456467  ...     -1.441646   
           ADM              0.143165           -0.951449  ...     -0.652047   
...                              ...                 ...  ...           ...   
           XEL             -0.938890            0.482821  ...     -1.524103   
           XOM              1.746691            0.140917  ...     -2.080909   
           YUM             -1.271440            1.703733  ...     -2.533718   
           ZBH             -0.467938           -0.348457  ...     -0.761665   
           ZBRA            -0.621100            0.173494  ...     -0.643108   

                   max_price_15d  max_price_21d       rsi  bb_lower  bb_upper  \
date       ticker                                                               
2010-02-22 A           -1.622441      -1.660897  1.054291  0.995363 -0.970205   
           ACGL        -1.690467      -1.680002  0.378266  0.452331 -0.631467   
           ACN         -1.616882      -1.590894 -1.015514 -0.732887 -0.034553   
           ADI         -1.447294      -1.492350  0.919647  2.052763 -1.320108   
           ADM         -0.528654      -0.603760 -0.538155 -0.582187 -0.188531   
...                          ...            ...       ...       ...       ...   
           XEL         -1.527166      -1.554176  0.275264  0.804388 -0.392609   
           XOM         -2.092752      -2.103439 -0.574371 -0.433733 -0.393669   
           YUM         -2.436433      -2.426964 -0.782901 -0.231202  0.465123   
           ZBH         -0.785743      -0.687300 -0.382754  0.453881  0.149595   
           ZBRA        -0.664945      -0.655987  0.845154  1.443538 -0.295454   

                   avg_true_range      macd  macd_hist  macd_signal

### Missing Test Entry

In [73]:
missing_test_entry_date=model1_test_X.index.get_level_values("date")[0]

In [74]:
non_stat_features_test_median=(model1_test_X
                               .groupby("ticker")[non_stationary_features]
                               .apply(lambda x: x.median(axis=0)))

In [75]:
missing_test_entry=model1_test_X.loc[idx[missing_test_entry_date,:]]

In [76]:
missing_test_entry[non_stationary_features]=(
    missing_test_entry[non_stationary_features]-
    non_stat_features_test_median
)

In [77]:
missing_test_entry["date"]=missing_test_entry_date
missing_test_entry.reset_index(inplace=True)
missing_test_entry.set_index(["date","ticker"],inplace=True)

In [78]:
missing_test_entry

open      high       low     close    volume  \
date       ticker                                                     
2016-02-24 A      -1.754108 -1.681043 -1.709934 -1.638476 -1.004473   
           ACGL   -1.347674 -1.301045 -1.353452 -1.270911 -0.671860   
           ACN    -1.076936 -1.062041 -1.079887 -1.031464 -0.293128   
           ADI    -2.223488 -2.139989 -2.219849 -2.111149 -0.093242   
           ADM    -1.193455 -1.185309 -1.216482 -1.159557 -0.443225   
...                     ...       ...       ...       ...       ...   
           XEL    -0.707696 -0.706594 -0.724697 -0.715449 -0.044058   
           XOM    -0.594392 -0.475619 -0.577104 -0.459759 -0.359788   
           YUM    -1.488278 -1.442949 -1.555714 -1.435785 -0.177071   
           ZBH    -1.138461 -1.123931 -1.139509 -1.108613 -0.713757   
           ZBRA   -0.742105 -0.676225 -0.762793 -0.656720  1.887575   

                   dollar_volume  dollar_volume_7d  dollar_volume_15d  \
date       ticker                                                       
2016-02-24 A           -0.848841         -0.147209           0.420747   
           ACGL        -0.238001          0.011808           0.482042   
           ACN          0.207714          0.665976           1.808727   
           ADI          0.492976          0.869604           0.781562   
           ADM         -0.397812         -0.413787           0.365349   
...                          ...               ...                ...   
           XEL          0.838983          1.651396           3.189286   
           XOM         -0.223299         -0.138214           0.846593   
           YUM          0.068456          0.522373           1.490927   
           ZBH         -0.440685         -0.158980           1.146670   
           ZBRA         1.758431          1.489330           1.494348   

                   dollar_volume_21d  dollar_volume_rank  ...  max_price_7d  \
date       ticker                                         ...                 
2016-02-24 A                0.198408            1.889614  ...     -1.608151   
           ACGL             0.668403            1.225129  ...     -1.311627   
           ACN              1.960762           -0.987460  ...     -1.095839   
           ADI              0.913982            1.296774  ...     -2.086624   
           ADM              0.935673            1.511759  ...     -1.229759   
...                              ...                 ...  ...           ...   
           XEL              3.279069           -1.736757  ...     -0.778809   
           XOM              0.884845           -0.382588  ...     -0.453267   
           YUM              1.629859           -0.219841  ...     -1.406805   
           ZBH              1.593811            0.375966  ...     -1.154456   
           ZBRA             1.509344           -0.243752  ...     -0.663233   

                   max_price_15d  max_price_21d       rsi  bb_lower  bb_upper  \
date       ticker                                                               
2016-02-24 A           -1.673263      -1.676020 -0.143330  0.219236 -0.378766   
           ACGL        -1.292990      -1.284986  0.027934  0.421950 -0.474066   
           ACN         -1.001297      -0.849125 -0.389882  0.789998  0.993036   
           ADI         -2.062492      -1.946578  0.003730  0.580436 -0.130064   
           ADM         -1.242922      -1.110585 -0.001634  0.329426 -0.156335   
...                          ...            ...       ...       ...       ...   
           XEL         -0.828717      -0.867488  1.318091  1.077082 -0.342368   
           XOM         -0.504273      -0.536538  0.586079  1.728172  0.144088   
           YUM         -1.356970      -1.299793  0.107561  0.904224  0.138182   
           ZBH         -1.137655      -0.953583 -0.764954  0.068151  0.823872   
           ZBRA        -0.680319      -0.715814  1.033493  3.685917 -0.625492   

                   avg_true_range      macd  macd_hist  macd_signal

### Missing Entries

In [82]:
missing_entries=pd.concat([missing_train_entry,missing_test_entry],axis=0)

In [83]:
missing_entries

open      high       low     close    volume  \
date       ticker                                                     
2010-02-22 A      -1.461939 -1.511306 -1.462434 -1.482303 -0.156541   
           ACGL   -1.689435 -1.692413 -1.671041 -1.689585  3.054903   
           ACN    -1.579930 -1.597301 -1.588901 -1.596231 -0.338131   
           ADI    -1.392832 -1.401546 -1.400281 -1.395146  0.904611   
           ADM    -0.602551 -0.608262 -0.582109 -0.597555 -0.511968   
...                     ...       ...       ...       ...       ...   
2016-02-24 XEL    -0.707696 -0.706594 -0.724697 -0.715449 -0.044058   
           XOM    -0.594392 -0.475619 -0.577104 -0.459759 -0.359788   
           YUM    -1.488278 -1.442949 -1.555714 -1.435785 -0.177071   
           ZBH    -1.138461 -1.123931 -1.139509 -1.108613 -0.713757   
           ZBRA   -0.742105 -0.676225 -0.762793 -0.656720  1.887575   

                   dollar_volume  dollar_volume_7d  dollar_volume_15d  \
date       ticker                                                       
2010-02-22 A           -0.693067         -0.279116          -0.341175   
           ACGL         1.343749          1.911222           1.708494   
           ACN         -0.758531         -0.957332          -0.950981   
           ADI         -0.044209          1.348163           0.612157   
           ADM         -0.916221         -0.615638           0.118269   
...                          ...               ...                ...   
2016-02-24 XEL          0.838983          1.651396           3.189286   
           XOM         -0.223299         -0.138214           0.846593   
           YUM          0.068456          0.522373           1.490927   
           ZBH         -0.440685         -0.158980           1.146670   
           ZBRA         1.758431          1.489330           1.494348   

                   dollar_volume_21d  dollar_volume_rank  ...  max_price_7d  \
date       ticker                                         ...                 
2010-02-22 A               -0.708899            0.014026  ...     -1.578164   
           ACGL             1.505118           -2.431023  ...     -1.706761   
           ACN             -0.963062            1.277944  ...     -1.613072   
           ADI              0.524131           -1.456467  ...     -1.441646   
           ADM              0.143165           -0.951449  ...     -0.652047   
...                              ...                 ...  ...           ...   
2016-02-24 XEL              3.279069           -1.736757  ...     -0.778809   
           XOM              0.884845           -0.382588  ...     -0.453267   
           YUM              1.629859           -0.219841  ...     -1.406805   
           ZBH              1.593811            0.375966  ...     -1.154456   
           ZBRA             1.509344           -0.243752  ...     -0.663233   

                   max_price_15d  max_price_21d       rsi  bb_lower  bb_upper  \
date       ticker                                                               
2010-02-22 A           -1.622441      -1.660897  1.054291  0.995363 -0.970205   
           ACGL        -1.690467      -1.680002  0.378266  0.452331 -0.631467   
           ACN         -1.616882      -1.590894 -1.015514 -0.732887 -0.034553   
           ADI         -1.447294      -1.492350  0.919647  2.052763 -1.320108   
           ADM         -0.528654      -0.603760 -0.538155 -0.582187 -0.188531   
...                          ...            ...       ...       ...       ...   
2016-02-24 XEL         -0.828717      -0.867488  1.318091  1.077082 -0.342368   
           XOM         -0.504273      -0.536538  0.586079  1.728172  0.144088   
           YUM         -1.356970      -1.299793  0.107561  0.904224  0.138182   
           ZBH         -1.137655      -0.953583 -0.764954  0.068151  0.823872   
           ZBRA        -0.680319      -0.715814  1.033493  3.685917 -0.625492   

                   avg_true_range      macd  macd_hist  macd_signal

### Complete Dataset

In [84]:
model8_X=pd.concat([model8_X,missing_entries],axis=0).sort_index()

# Computing Model Factors

## Model-1 Factors

In [98]:
factors=model1.predict(model1_X)

In [100]:
model1_factors=pd.DataFrame(
    {"Model1_Factors":factors},
    index=pd.MultiIndex.from_arrays(
        [
            model1_X.index.get_level_values("date"),
            model1_X.index.get_level_values("ticker") 
        ],
        names=["date","ticker"]
    )
)

In [101]:
model1_factors

Model1_Factors
date       ticker                
2010-02-22 A            -0.001913
           ACGL          0.000990
           ACN           0.001405
           ADI          -0.003231
           ADM           0.002703
...                           ...
2017-11-29 XEL          -0.017937
           XOM          -0.009932
           YUM          -0.015115
           ZBH          -0.004992
           ZBRA         -0.011484

[732292 rows x 1 columns]

## Model-4 Factors

In [102]:
factors=model4.predict(model4_X)

In [103]:
model4_factors=pd.DataFrame(
    {"Model4_Factors":factors},
    index=pd.MultiIndex.from_arrays(
        [
            model4_X.index.get_level_values("date"),
            model4_X.index.get_level_values("ticker") 
        ],
        names=["date","ticker"]
    )
)

In [104]:
model4_factors

Model4_Factors
date       ticker                
2010-02-22 A            -0.002870
           ACGL         -0.000440
           ACN           0.000259
           ADI          -0.003648
           ADM           0.001880
...                           ...
2017-11-29 XEL          -0.009582
           XOM          -0.002927
           YUM          -0.009351
           ZBH          -0.002113
           ZBRA         -0.001230

[732292 rows x 1 columns]

## Model-2 Factors

In [105]:
factors=model2.predict(model2_X)

In [106]:
model2_factors=pd.DataFrame(
    {"Model2_Factors":factors},
    index=pd.MultiIndex.from_arrays(
        [
            model2_X.index.get_level_values("date"),
            model2_X.index.get_level_values("ticker") 
        ],
        names=["date","ticker"]
    )
)

In [107]:
model2_factors

Model2_Factors
date       ticker                
2010-02-22 A            -0.002329
           ACGL         -0.000315
           ACN           0.001138
           ADI          -0.004163
           ADM           0.002698
...                           ...
2017-11-29 XEL          -0.017144
           XOM          -0.010068
           YUM          -0.015095
           ZBH          -0.004722
           ZBRA         -0.011580

[732292 rows x 1 columns]

## Model-3 Factors

In [108]:
factors=model3.predict(model3_X)

In [109]:
model3_factors=pd.DataFrame(
    {"Model3_Factors":factors},
    index=pd.MultiIndex.from_arrays(
        [
            model3_X.index.get_level_values("date"),
            model3_X.index.get_level_values("ticker") 
        ],
        names=["date","ticker"]
    )
)

In [110]:
model3_factors

Model3_Factors
date       ticker                
2010-02-22 A            -0.002245
           ACGL          0.000595
           ACN           0.001061
           ADI          -0.004417
           ADM          -0.000160
...                           ...
2017-11-29 XEL          -0.016153
           XOM          -0.010261
           YUM          -0.010268
           ZBH          -0.009067
           ZBRA         -0.012544

[732292 rows x 1 columns]

## Model-8 Factors

In [111]:
factors=model8.predict(model8_X)

In [112]:
model8_factors=pd.DataFrame(
    {"Model8_Factors":factors},
    index=pd.MultiIndex.from_arrays(
        [
            model8_X.index.get_level_values("date"),
            model8_X.index.get_level_values("ticker") 
        ],
        names=["date","ticker"]
    )
)

In [113]:
model8_factors

Model8_Factors
date       ticker                
2010-02-22 A            -0.094332
           ACGL         -0.091649
           ACN          -0.090515
           ADI          -0.085072
           ADM          -0.028546
...                           ...
2017-11-29 XEL          -0.012405
           XOM          -0.005234
           YUM          -0.002357
           ZBH           0.004256
           ZBRA         -0.016112

[732292 rows x 1 columns]

# Combined Factor

In [114]:
factors=pd.concat([
    model1_factors,
    model4_factors,
    model2_factors,
    model3_factors,
    model8_factors,
    
],axis=1)

In [115]:
factors

Model1_Factors  Model4_Factors  Model2_Factors  \
date       ticker                                                   
2010-02-22 A            -0.001913       -0.002870       -0.002329   
           ACGL          0.000990       -0.000440       -0.000315   
           ACN           0.001405        0.000259        0.001138   
           ADI          -0.003231       -0.003648       -0.004163   
           ADM           0.002703        0.001880        0.002698   
...                           ...             ...             ...   
2017-11-29 XEL          -0.017937       -0.009582       -0.017144   
           XOM          -0.009932       -0.002927       -0.010068   
           YUM          -0.015115       -0.009351       -0.015095   
           ZBH          -0.004992       -0.002113       -0.004722   
           ZBRA         -0.011484       -0.001230       -0.011580   

                   Model3_Factors  Model8_Factors  
date       ticker                                  
2010-02-22 A            -0.002245       -0.094332  
           ACGL          0.000595       -0.091649  
           ACN           0.001061       -0.090515  
           ADI          -0.004417       -0.085072  
           ADM          -0.000160       -0.028546  
...                           ...             ...  
2017-11-29 XEL          -0.016153       -0.012405  
           XOM          -0.010261       -0.005234  
           YUM          -0.010268       -0.002357  
           ZBH          -0.009067        0.004256  
           ZBRA         -0.012544       -0.016112  

[732292 rows x 5 columns]

In [117]:
factors["Combined_Factor"]=factors.mean(axis=1)

In [118]:
factors

Model1_Factors  Model4_Factors  Model2_Factors  \
date       ticker                                                   
2010-02-22 A            -0.001913       -0.002870       -0.002329   
           ACGL          0.000990       -0.000440       -0.000315   
           ACN           0.001405        0.000259        0.001138   
           ADI          -0.003231       -0.003648       -0.004163   
           ADM           0.002703        0.001880        0.002698   
...                           ...             ...             ...   
2017-11-29 XEL          -0.017937       -0.009582       -0.017144   
           XOM          -0.009932       -0.002927       -0.010068   
           YUM          -0.015115       -0.009351       -0.015095   
           ZBH          -0.004992       -0.002113       -0.004722   
           ZBRA         -0.011484       -0.001230       -0.011580   

                   Model3_Factors  Model8_Factors  Combined_Factor  
date       ticker                                                   
2010-02-22 A            -0.002245       -0.094332        -0.020738  
           ACGL          0.000595       -0.091649        -0.018164  
           ACN           0.001061       -0.090515        -0.017330  
           ADI          -0.004417       -0.085072        -0.020106  
           ADM          -0.000160       -0.028546        -0.004285  
...                           ...             ...              ...  
2017-11-29 XEL          -0.016153       -0.012405        -0.014644  
           XOM          -0.010261       -0.005234        -0.007684  
           YUM          -0.010268       -0.002357        -0.010437  
           ZBH          -0.009067        0.004256        -0.003328  
           ZBRA         -0.012544       -0.016112        -0.010590  

[732292 rows x 6 columns]

# Factor Statistics

In [123]:
percentiles=[0.001,0.005,0.01,0.1,0.25]
percentiles+=[1-p for p in percentiles]
factors.describe(percentiles)

,Model1_Factors,Model4_Factors,Model2_Factors,Model3_Factors,Model8_Factors,Combined_Factor
count,732292.000000,732292.000000,732292.000000,732292.000000,732292.000000,732292.000000
mean,-0.003564,-0.001843,-0.003580,-0.003412,-0.001914,-0.002862
std,0.007951,0.006969,0.007795,0.007063,0.007789,0.006920
min,-0.347578,-0.342686,-0.352665,-0.370912,-0.904161,-0.340408
0.1%,-0.044123,-0.040464,-0.043728,-0.039475,-0.052999,-0.039918
0.5%,-0.028552,-0.025552,-0.028147,-0.024757,-0.027788,-0.025256
1%,-0.024028,-0.021128,-0.023750,-0.020835,-0.022317,-0.021076
10%,-0.012599,-0.009679,-0.012433,-0.010931,-0.010110,-0.010543
25%,-0.007892,-0.005374,-0.007810,-0.006933,-0.005646,-0.006489
50%,-0.003279,-0.001474,-0.003280,-0.003106,-0.001542,-0.002604


In [124]:
factors.corr()

,Model1_Factors,Model4_Factors,Model2_Factors,Model3_Factors,Model8_Factors,Combined_Factor
Model1_Factors,1.000000,0.924026,0.987657,0.738907,0.849338,0.980509
Model4_Factors,0.924026,1.000000,0.905647,0.611401,0.866762,0.937766
Model2_Factors,0.987657,0.905647,1.000000,0.761164,0.830302,0.977009
Model3_Factors,0.738907,0.611401,0.761164,1.000000,0.577931,0.798710
Model8_Factors,0.849338,0.866762,0.830302,0.577931,1.000000,0.899950
Combined_Factor,0.980509,0.937766,0.977009,0.798710,0.899950,1.000000


# Saving Factors

In [125]:
factors.to_csv(
    os.path.join(data_store,"factors.csv"),
    index=True
)